In [1]:
import logging
import os

os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'
from napistu.genomics.scverse_loading import DatasetsConfig

from napistu_torch.load.constants import FM_DEFS, SCGPT_DEFS
from napistu_torch.load.foundation_models import FoundationModel
from napistu_torch.load.foundation_model_etl import process_scgpt

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


In [ ]:
# Configuration
DATA_DIR = "data"
OUTPUT_DIR = "output"
# Raw config dictionary
DATASETS_CONFIG = {
    "efthymiou2025": {
        "uri": "https://cellxgene.cziscience.com/collections/6b701826-37bb-4356-9792-ff41fc4c3161",
        "path": os.path.expanduser("~/Desktop/DATA/genomics/efthymiou.h5ad")
    }
}

MODEL_PATH = os.path.join(DATA_DIR, "scGPT_bc")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Validated config
datasets_config = DatasetsConfig(DATASETS_CONFIG)


In [ ]:
process_scgpt(MODEL_PATH, OUTPUT_DIR, datasets_config=datasets_config)


In [ ]:
x = foundation_model.dataset_gene_embeddings["efthymiou2025"].get('scGPT/efthymiou2025/adipocyte (0)')
x.category

In [ ]:
# Load FoundationModel
foundation_model = FoundationModel.load(OUTPUT_DIR, SCGPT_DEFS.MODEL_NAME)
embeddings = foundation_model.weights.static_gene_embeddings

GENES_OF_INTEREST = embeddings.gene_annotations[FM_DEFS.VOCAB_NAME].sample(20000).tolist()
GENE_MASK = [x in GENES_OF_INTEREST for x in embeddings.ordered_gene_ids]

# Compute attention on demand using FoundationModelWeights method
# This handles multi-head attention properly
layer_11_attn = foundation_model.weights.compute_attention_from_weights(
    layer_idx=11,
    n_heads=foundation_model.n_heads,
    gene_mask=GENE_MASK
)

foundation_model.dataset_gene_embeddings
foundation_model.weights.static_gene_embeddings

In [2]:
import os

import torch
from napistu.genomics.scverse_loading import DatasetsConfig
from napistu_torch.load.constants import FM_DEFAULTS
from napistu_torch.load.foundation_model_etl import (
    _scgpt_load_model,
    _scgpt_load_gene_annotations,
    _scgpt_preprocess_dataset,
    _get_cell_clusters_and_category_dict,
)
from napistu_torch.utils.torch_utils import ensure_device

# --- Config (same as your notebook) ---
DATA_DIR = "data"
MODEL_PATH = f"{DATA_DIR}/scGPT_bc"
ANNOTATIONS_PATH = f"{DATA_DIR}/scgpt_gene_info.csv"

DATASETS_CONFIG = {
    "efthymiou2025": {
        "uri": "https://cellxgene.cziscience.com/collections/6b701826-37bb-4356-9792-ff41fc4c3161",
        "path": os.path.expanduser("~/Desktop/DATA/genomics/efthymiou.h5ad"),
    }
}
datasets_config = DatasetsConfig(DATASETS_CONFIG)
dataset_config = datasets_config.data["efthymiou2025"]

MIN_CLUSTER_CELLS = FM_DEFAULTS.MIN_CLUSTER_CELLS

# --- Load model, vocab, annotations, data ---
model, vocab, model_metadata, checkpoint_path = _scgpt_load_model(MODEL_PATH)
gene_annotations = _scgpt_load_gene_annotations(ANNOTATIONS_PATH)
adata = dataset_config.load_h5ad()

device = ensure_device(None, allow_autoselect=True)
model = model.to(device)
model.eval()

# --- Replicate _scgpt_get_gene_embedding_by_cell_type up to the batch call ---

# 1. Cluster bookkeeping (uses raw adata.obs, same as ETL)
cell_clusters, cell_cluster_dict = _get_cell_clusters_and_category_dict(
    adata.obs, min_cluster_cells=MIN_CLUSTER_CELLS
)
print(f"Found {len(cell_clusters)} clusters")

# 2. Preprocess: HVG selection, binning, gets gene indices into vocab
adata_subset, selected_genes, gene_indices = _scgpt_preprocess_dataset(
    adata, model, vocab, gene_annotations
)
print(f"Selected {len(selected_genes)} HVGs, adata_subset shape: {adata_subset.shape}")

# 3. Static gene embeddings for the selected genes
with torch.no_grad():
    d_gene_indices = gene_indices.to(device)
    gene_embeddings = model.encoder(d_gene_indices)  # (n_genes, embed_dim)
    del d_gene_indices

# 4. Pick a single cluster to work with
#    Using the first cluster here; swap in any leiden_scVI value you want
cluster_row = cell_clusters.iloc[0]
cluster_id = cluster_row["leiden_scVI"]
cluster_name = cell_cluster_dict[0]
print(f"Working with cluster: {cluster_name} (leiden={cluster_id})")

cluster_mask = adata_subset.obs["leiden_scVI"] == cluster_id
cluster_adata = adata_subset[cluster_mask]
print(f"  {cluster_adata.shape[0]} cells")

# 5. Expression tensor for this cluster
if model.input_emb_style == "continuous":
    cluster_expr = torch.tensor(cluster_adata.X, dtype=torch.float32)
else:
    cluster_expr = torch.tensor(cluster_adata.X)

# --- At this point you have everything _embed_expression_batch takes ---
# model, model_type=SCGPT, expression=cluster_expr, gene_emb=gene_embeddings,
# gene_indices=gene_indices, batch_size=64, device=device

print("\nReady for _embed_expression_batch:")
print(f"  expression: {cluster_expr.shape}")
print(f"  gene_embeddings: {gene_embeddings.shape}")
print(f"  gene_indices: {gene_indices.shape}")
print(f"  device: {device}")


INFO:datasets:PyTorch version 2.3.0 available.
INFO:datasets:JAX version 0.7.2 available.


Resume model from data/scGPT_bc/best_model.pt, the model args will override the config data/scGPT_bc/args.json.


INFO:napistu_torch.load.foundation_model_etl:Loaded 159/163 parameters from checkpoint. Skipped 4 keys not present in the vanilla model (likely auxiliary heads): ['flag_encoder.weight', 'mvc_decoder.W.weight', 'mvc_decoder.gene2query.bias', 'mvc_decoder.gene2query.weight']
INFO:napistu_torch.load.foundation_model_etl:Loaded scGPT weights from data/scGPT_bc/best_model.pt
INFO:napistu_torch.load.foundation_model_etl:Found 35467 common genes between data and model vocab


Found 19 clusters


INFO:napistu_torch.load.foundation_model_etl:Model input_style: continuous


scGPT - INFO - Filtering cells by counts ...
scGPT - INFO - Normalizing total counts ...
scGPT - INFO - Subsetting highly variable genes ...
scGPT - WARNING - No batch_key is provided, will use all cells for HVG selection.
scGPT - INFO - Binning data ...
scGPT - WARNING - The input data contains all zero rows. Please make sure this is expected. You can use the `filter_cell_by_counts` arg to filter out all zero rows.
scGPT - WARNING - The input data contains all zero rows. Please make sure this is expected. You can use the `filter_cell_by_counts` arg to filter out all zero rows.


INFO:napistu_torch.load.foundation_model_etl:Selected 1200 HVGs for processing


Selected 1200 HVGs, adata_subset shape: (72328, 1200)
Working with cluster: adipocyte (0) (leiden=0)
  12365 cells

Ready for _embed_expression_batch:
  expression: torch.Size([12365, 1200])
  gene_embeddings: torch.Size([1200, 512])
  gene_indices: torch.Size([1200])
  device: mps


In [3]:
from napistu_torch.load.foundation_model_etl import _scgpt_capture_residual_streams_per_cluster

stream = _scgpt_capture_residual_streams_per_cluster(
    model=model,
    expression=cluster_expr,
    gene_indices=gene_indices,
    batch_size=64,
    device=device,
)

In [10]:

import torch

# ============================================================
# Check 1: Hooks don't perturb the forward pass
# ============================================================
# Run a small batch with no hooks, compare to a hooked forward.
# The capture function removes hooks in its finally block, so right
# now `model` has no hooks registered — we run the clean pass first.

small_expr = cluster_expr[:4].to(device)
bsz, n_genes = small_expr.shape
src = gene_indices.to(device).unsqueeze(0).expand(bsz, -1)

with torch.no_grad():
    clean_output = model._encode(src, small_expr, None, batch_labels=None)
clean_output = clean_output.detach().cpu()

# Now run the same 4 cells through the hooked capture function
hooked_stream = _scgpt_capture_residual_streams_per_cluster(
    model=model,
    expression=cluster_expr[:4],
    gene_indices=gene_indices,
    batch_size=64,
    device=device,
)

# Re-run the clean forward after hooks have been removed — should match
with torch.no_grad():
    post_hook_output = model._encode(src, small_expr, None, batch_labels=None)
post_hook_output = post_hook_output.detach().cpu()

print("Check 1 — forward pass unchanged by hook lifecycle:")
print(f"  clean vs post-hook-removal: {torch.allclose(clean_output, post_hook_output)}")

# ============================================================
# Check 2: Layer-0 residual ≈ static + expression encoding
# ============================================================
# The layer-0 residual stream is what transformer_encoder sees as input,
# which is the sum of static gene embeddings and expression encoding.
# Recompute this manually for one cell and compare to stream[0] averaged
# over that single cell.

# Grab one cell, run hook-based capture, compare to manual reconstruction
single_cell_stream = _scgpt_capture_residual_streams_per_cluster(
    model=model,
    expression=cluster_expr[:1],
    gene_indices=gene_indices,
    batch_size=1,
    device=device,
)
# single_cell_stream shape: (12, 1200, 512) — already averaged over 1 cell

# Manual reconstruction: static gene embedding + expression encoding
# For scGPT, _encode combines these. We need to mimic what _encode does
# up to the point transformer_encoder is called.
single_expr = cluster_expr[:1].to(device)  # (1, 1200)
src = gene_indices.to(device).unsqueeze(0)  # (1, 1200)

with torch.no_grad():
    # This replicates the static + expression combination as scGPT does it.
    # Exact API depends on scGPT version. The pieces are:
    #   - model.encoder(src): static gene token embedding
    #   - model.value_encoder(expression): expression component (continuous path)
    # For binned input, scGPT uses a different encoder — inspect model.value_encoder
    # to see which one applies to your checkpoint.
    static_part = model.encoder(src)              # (1, 1200, 512)
    expr_part = model.value_encoder(single_expr)  # (1, 1200, 512)
    manual_layer_0 = (static_part + expr_part).squeeze(0).cpu()

captured_layer_0 = single_cell_stream[0]  # (1200, 512)

diff = (manual_layer_0 - captured_layer_0).abs()
print("\nCheck 2 — layer 0 capture matches static + expression encoding:")
print(f"  max abs diff: {diff.max():.6f}")
print(f"  mean abs diff: {diff.mean():.6f}")
print(f"  allclose (atol=1e-4): {torch.allclose(manual_layer_0, captured_layer_0, atol=1e-4)}")

# ============================================================
# Check 3: Layers show meaningful variation (fast diagnostic)
# ============================================================
print(f"\nCheck 3 — per-layer statistics for cluster '{cluster_name}':")
print(f"{'layer':>6}  {'mean':>10}  {'std':>10}  {'norm':>12}")
for i in range(stream.shape[0]):
    layer_i = stream[i]
    print(
        f"{i:>6}  {layer_i.mean().item():>10.4f}  "
        f"{layer_i.std().item():>10.4f}  {layer_i.norm().item():>12.2f}"
    )

Check 1 — forward pass unchanged by hook lifecycle:
  clean vs post-hook-removal: True

Check 2 — layer 0 capture matches static + expression encoding:
  max abs diff: 0.000000
  mean abs diff: 0.000000
  allclose (atol=1e-4): True

Check 3 — per-layer statistics for cluster 'adipocyte (0)':
 layer        mean         std          norm
     0      0.0115      1.1584        908.03
     1      0.0040      0.9239        724.20
     2      0.0041      0.8963        702.56
     3      0.0038      0.9145        716.82
     4      0.0039      0.9304        729.29
     5      0.0030      0.9435        739.53
     6      0.0014      0.9473        742.53
     7      0.0003      0.9626        754.55
     8      0.0004      0.9820        769.70
     9     -0.0007      1.0179        797.83
    10     -0.0001      1.0570        828.50
    11      0.0062      1.1314        886.87
